In [1]:
# =========================================================
# GOOGLE COLAB + GOOGLE DRIVE SETUP
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# IMPORTS
# =========================================================
import os
import glob
import re
import pandas as pd
import numpy as np
import cv2
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import KFold
from PIL import Image

# =========================================================
# MAIN PATH CONFIGURATION
# =========================================================
DATASET_ROOT = "/content/drive/MyDrive/Deep_learning_data"

LOCAL_SAVE_PATH = "/content/universal_light_direction_model.pth"
DRIVE_SAVE_PATH = os.path.join(DATASET_ROOT, "universal_light_direction_model.pth")

# Unified multi-dataset configuration optimized strictly for RGB
ALL_DATASETS_CONFIG = {
    # Original Datasets
    "monkey": {
        "rgb_dir": "monkey_RGB_img",
        "labels_file": "monkey_labels.csv"
    },
    "robot": {
        "rgb_dir": "robot_RGB_img",
        "labels_file": "robot_labels.csv"
    },
    "snowman": {
        "rgb_dir": "new_snowman_RGB_img",
        "labels_file": "snowman_labels.csv"
    },
    # New Real-World Folders
    "flask": {
        "rgb_dir": "Flask/RGB",
        "labels_file": "Flask/light_directions.xlsx"
    },
    "cup": {
        "rgb_dir": "cup/RGB",
        "labels_file": "cup/light_directions.xlsx"
    },
    "handcream": {
        "rgb_dir": "Handcream/RGB",
        "labels_file": "Handcream/light_directions.xlsx"
    },
    "lipstick": {
        "rgb_dir": "Lipstick/RGB",
        "labels_file": "Lipstick/light_directions.xlsx"
    }
}

# =========================================================
# MODEL DEFINITION (Standard 3-Channel RGB ResNet18)
# =========================================================
class MultiModalRobustModel(nn.Module):
    def __init__(self, output_dim=3):
        super().__init__()
        # Standard 3-channel input ResNet-18
        self.encoder = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

        # Replace the native classifier with your custom multi-layer regression head
        self.encoder.fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )

    def forward(self, rgb):
        # Directly pass the 3-channel RGB image through the network
        return F.normalize(self.encoder(rgb), p=2, dim=1)

# =========================================================
# UNIVERSAL BALANCED DATASET CLASS (RGB ONLY)
# =========================================================
class UniversalMultiModalDataset(Dataset):
    def __init__(self, config_dict, train_mode=True):
        self.train_mode = train_mode
        self.all_samples = []

        print("====== Initializing Universal Dataset Pipeline ======")
        for domain_name, paths in config_dict.items():
            labels_p = os.path.normpath(os.path.join(DATASET_ROOT, paths["labels_file"]))
            if not os.path.exists(labels_p):
                print(f"Skipping domain '{domain_name}': File not found at {labels_p}")
                continue

            if labels_p.endswith('.xlsx') or labels_p.endswith('.xls'):
                df = pd.read_excel(labels_p)
            else:
                df = pd.read_csv(labels_p)

            df.columns = [str(c).strip().lower() for c in df.columns]

            possible_file_cols = ['filename', 'file_name', 'image', 'img', 'name', 'image_name', 'imagename']
            file_col = next((c for c in possible_file_cols if c in df.columns), df.columns[0])

            light_cols = None
            possible_triplets = [
                ['light_x', 'light_y', 'light_z'],
                ['l_x', 'l_y', 'l_z'],
                ['light_dir_x', 'light_dir_y', 'light_dir_z'],
                ['light_direction_x', 'light_direction_y', 'light_direction_z']
            ]

            for triplet in possible_triplets:
                if all(c in df.columns for c in triplet):
                    light_cols = triplet
                    break

            if light_cols is None:
                light_cols = list(df.columns[-3:])

            domain_count = 0
            for _, row in df.iterrows():
                self.all_samples.append({
                    "domain": domain_name,
                    "filename": str(row[file_col]).strip(),
                    "target": [float(row[c]) for c in light_cols],
                    "paths": paths
                })
                domain_count += 1
            print(f"Loaded {domain_count} samples from Domain: [{domain_name}]")

        print(f"Combined Dataset Construction Completed. Total Mixed Samples: {len(self.all_samples)}\n")

    def find_image_path(self, folder, filename):
        exact = os.path.join(folder, filename)
        if os.path.exists(exact):
            return exact

        base_name = os.path.splitext(filename)[0]
        num_match = re.search(r'\d+', base_name)
        num_str = num_match.group(0) if num_match else None

        candidates = [filename, base_name]
        for ext in ['jpg', 'jpeg', 'png', 'JPG', 'JPEG', 'PNG']:
            candidates.append(f"{base_name}.{ext}")
            if num_str:
                clean_base = base_name.replace(num_str, str(int(num_str)))
                candidates.append(f"{clean_base}.{ext}")
                candidates.append(f"{clean_base.replace(str(int(num_str)), f'{int(num_str):04d}')}.{ext}")

        for cand in set(candidates):
            cand_path = os.path.join(folder, cand)
            if os.path.exists(cand_path):
                return cand_path

        if os.path.exists(folder):
            all_files = os.listdir(folder)
            for f in all_files:
                if base_name.lower() in f.lower():
                    return os.path.join(folder, f)
            if num_str:
                for f in all_files:
                    f_num = re.search(r'\d+', f)
                    if f_num and int(f_num.group(0)) == int(num_str):
                        return os.path.join(folder, f)

        raise FileNotFoundError(f"Missing sample file: {filename} inside folder context: {folder}")

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        domain = sample["domain"]
        filename = sample["filename"]
        paths = sample["paths"]

        rgb_dir = os.path.normpath(os.path.join(DATASET_ROOT, paths["rgb_dir"]))
        rgb_p = self.find_image_path(rgb_dir, filename)
        rgb_img = Image.open(rgb_p).convert('RGB').resize((224, 224))

        target_coords = list(sample["target"])

        if self.train_mode:
            if np.random.rand() > 0.5:
                rgb_img = TF.hflip(rgb_img)
                target_coords[0] = -target_coords[0]

            if np.random.rand() > 0.5:
                angle = float(np.random.uniform(-20, 20))
                rgb_img = TF.rotate(rgb_img, angle)

            if np.random.rand() > 0.5:
                color_jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
                rgb_img = color_jitter(rgb_img)

        rgb_tensor = TF.to_tensor(rgb_img)
        rgb_tensor = TF.normalize(rgb_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        target_tensor = torch.tensor(target_coords, dtype=torch.float32)

        return rgb_tensor, target_tensor, domain

# =========================================================
# ENGINE TRAINING LOOPS (RGB-Only)
# =========================================================
def run_universal_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Execution Target Device: {device}")

    full_dataset = UniversalMultiModalDataset(ALL_DATASETS_CONFIG, train_mode=True)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    best_overall_val_loss = float('inf')

    for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset)):
        print(f"\n==========================================")
        print(f"       STARTING RGB CV FOLD {fold+1}/5        ")
        print(f"==========================================")

        train_sub = torch.utils.data.Subset(full_dataset, train_idx)
        val_sub = torch.utils.data.Subset(full_dataset, val_idx)

        train_sub.dataset.train_mode = True
        val_sub.dataset.train_mode = False

        train_loader = DataLoader(train_sub, batch_size=16, shuffle=True, num_workers=0, drop_last=True)
        val_loader = DataLoader(val_sub, batch_size=16, shuffle=False, num_workers=0)

        model = MultiModalRobustModel(output_dim=3).to(device)

        optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
        criterion = nn.MSELoss()

        epochs = 12
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0

            for rgb, targets, _ in train_loader:
                rgb, targets = rgb.to(device), targets.to(device)

                optimizer.zero_grad()
                outputs = model(rgb)
                loss = criterion(outputs, targets)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                running_loss += loss.item()

            avg_train_loss = running_loss / len(train_loader)

            model.eval()
            domain_losses = {name: [] for name in ALL_DATASETS_CONFIG.keys()}
            total_val_loss = 0.0

            with torch.no_grad():
                for rgb, targets, batch_domains in val_loader:
                    rgb, targets = rgb.to(device), targets.to(device)
                    preds = model(rgb)

                    losses = torch.mean((preds - targets) ** 2, dim=1)
                    total_val_loss += losses.mean().item()

                    for i, d_name in enumerate(batch_domains):
                        if d_name in domain_losses:
                            domain_losses[d_name].append(losses[i].item())

            avg_val_loss = total_val_loss / len(val_loader)

            print(f"Fold {fold+1} | Epoch [{epoch+1:02d}/{epochs}] | Overall Train Loss: {avg_train_loss:.4f} | Overall Val Loss: {avg_val_loss:.4f}")
            domain_str = "   ↳ Val breakdown -> "
            for d_name, d_list in domain_losses.items():
                if d_list:
                    domain_str += f"{d_name}: {np.mean(d_list):.4f} | "
            print(domain_str[:-3])

        if avg_val_loss < best_overall_val_loss:
            best_overall_val_loss = avg_val_loss
            torch.save(model.state_dict(), LOCAL_SAVE_PATH)
            print(f"--> Saved current best global model structure to local cache!")

            try:
                shutil.copy2(LOCAL_SAVE_PATH, DRIVE_SAVE_PATH)
                print(f"--> Successfully synchronized model out to permanent Drive location.")
            except Exception as e:
                print(f"--> Sync skipped due to volatile Drive write constraints.")

if __name__ == "__main__":
    run_universal_training()

Mounted at /content/drive
Execution Target Device: cpu
====== Initializing Universal Dataset Pipeline ======
Loaded 200 samples from Domain: [monkey]
Loaded 300 samples from Domain: [robot]
Loaded 300 samples from Domain: [snowman]
Loaded 100 samples from Domain: [flask]
Loaded 86 samples from Domain: [cup]
Loaded 91 samples from Domain: [handcream]
Loaded 100 samples from Domain: [lipstick]
Combined Dataset Construction Completed. Total Mixed Samples: 1177


       STARTING RGB CV FOLD 1/5        
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 245MB/s]


Fold 1 | Epoch [01/12] | Overall Train Loss: 0.0951 | Overall Val Loss: 0.0262
   ↳ Val breakdown -> monkey: 0.0904 | robot: 0.0189 | snowman: 0.0121 | flask: 0.0113 | cup: 0.0190 | handcream: 0.0052 | lipstick: 0.0084
Fold 1 | Epoch [02/12] | Overall Train Loss: 0.0297 | Overall Val Loss: 0.0143
   ↳ Val breakdown -> monkey: 0.0472 | robot: 0.0116 | snowman: 0.0046 | flask: 0.0067 | cup: 0.0105 | handcream: 0.0055 | lipstick: 0.0065
Fold 1 | Epoch [03/12] | Overall Train Loss: 0.0182 | Overall Val Loss: 0.0099
   ↳ Val breakdown -> monkey: 0.0297 | robot: 0.0096 | snowman: 0.0048 | flask: 0.0036 | cup: 0.0053 | handcream: 0.0028 | lipstick: 0.0049
Fold 1 | Epoch [04/12] | Overall Train Loss: 0.0133 | Overall Val Loss: 0.0078
   ↳ Val breakdown -> monkey: 0.0243 | robot: 0.0067 | snowman: 0.0034 | flask: 0.0038 | cup: 0.0028 | handcream: 0.0019 | lipstick: 0.0059
Fold 1 | Epoch [05/12] | Overall Train Loss: 0.0123 | Overall Val Loss: 0.0071
   ↳ Val breakdown -> monkey: 0.0215 | robot:

In [4]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from PIL import Image

# =========================================================
# PATH CONFIGURATION
# =========================================================
DATASET_ROOT = "/content/drive/MyDrive/Deep_learning_data"
DRIVE_SAVE_PATH = os.path.join(DATASET_ROOT, "universal_light_direction_model.pth")

# Custom Evaluation split pointing to your new Apple RGB domain files
EVAL_APPLE_CONFIG = {
    "apple": {
        "rgb_dir": "Apple/RGB",
        "labels_file": "Apple/labels.csv"
    }
}

# =========================================================
# ANGULAR METRIC CALCULATOR
# =========================================================
def calculate_angular_error(v1, v2):
    """
    Computes true angular error in degrees between prediction and ground truth vectors.
    """
    v1_norm = F.normalize(v1, p=2, dim=1)
    v2_norm = F.normalize(v2, p=2, dim=1)

    dot_product = torch.sum(v1_norm * v2_norm, dim=1).clamp(-1.0, 1.0)
    radians = torch.acos(dot_product)
    return torch.rad2deg(radians)

# =========================================================
# EVALUATION DATASET CLASS (RGB MATCHED)
# =========================================================
class AppleEvalDataset(Dataset):
    def __init__(self, config_dict):
        self.all_samples = []

        for domain_name, paths in config_dict.items():
            labels_p = os.path.normpath(os.path.join(DATASET_ROOT, paths["labels_file"]))
            if not os.path.exists(labels_p):
                print(f"Error: Labels reference file missing at {labels_p}")
                continue

            df = pd.read_csv(labels_p)
            df.columns = [str(c).strip().lower() for c in df.columns]

            possible_file_cols = ['filename', 'file_name', 'image', 'img', 'name', 'image_name', 'imagename']
            file_col = next((c for c in possible_file_cols if c in df.columns), df.columns[0])

            light_cols = None
            possible_triplets = [
                ['light_x', 'light_y', 'light_z'],
                ['l_x', 'l_y', 'l_z'],
                ['light_dir_x', 'light_dir_y', 'light_dir_z']
            ]
            for triplet in possible_triplets:
                if all(c in df.columns for c in triplet):
                    light_cols = triplet
                    break
            if light_cols is None:
                light_cols = list(df.columns[-3:])

            for idx, row in df.iterrows():
                fname = str(row[file_col]).strip()

                # Manual index correction to match your absolute disk naming mapping
                if idx == 0:
                    fname = "Apple_1.jpg"
                elif idx == 1:
                    fname = "Apple_2.jpg"
                else:
                    break

                self.all_samples.append({
                    "domain": domain_name,
                    "filename": fname,
                    "target": [float(row[c]) for c in light_cols],
                    "paths": paths
                })

    def find_image_path(self, folder, filename):
        exact = os.path.join(folder, filename)
        if os.path.exists(exact):
            return exact
        base_name = os.path.splitext(filename)[0]
        all_files = os.listdir(folder)
        for f in all_files:
            if base_name.lower() in f.lower():
                return os.path.join(folder, f)
        raise FileNotFoundError(f"Missing sample file: {filename} inside folder: {folder}")

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        filename = sample["filename"]
        paths = sample["paths"]

        rgb_dir_path = os.path.normpath(os.path.join(DATASET_ROOT, paths["rgb_dir"]))
        rgb_p = self.find_image_path(rgb_dir_path, filename)

        rgb_img = Image.open(rgb_p).convert('RGB').resize((224, 224))
        rgb_tensor = TF.to_tensor(rgb_img)
        rgb_tensor = TF.normalize(rgb_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        target_tensor = torch.tensor(sample["target"], dtype=torch.float32)
        return rgb_tensor, target_tensor, filename, paths["rgb_dir"]

# =========================================================
# EVALUATION RUNNER ENGINE
# =========================================================
def run_apple_prediction_check():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Execution Device: {device}")

    # Process evaluation pipeline dataset loading split
    eval_dataset = AppleEvalDataset(EVAL_APPLE_CONFIG)
    if len(eval_dataset) == 0:
        print("Error: No evaluation entries loaded for the Apple dataset domain.")
        return

    # Instantiate your specific class architecture
    try:
        model = MultiModalRobustModel(output_dim=3).to(device)
    except NameError:
        print("Error: 'MultiModalRobustModel' class blueprint needs to be compiled before executing this check.")
        return

    # Synchronize trained baseline parameters
    if os.path.exists(DRIVE_SAVE_PATH):
        model.load_state_dict(torch.load(DRIVE_SAVE_PATH, map_location=device))
        print(f"Loaded trained baseline parameters from: {DRIVE_SAVE_PATH}")
    else:
        print(f"Weights parameters missing at destination: {DRIVE_SAVE_PATH}")
        return

    model.eval()

    print("\n" + "="*135)
    print(f" {'IMAGE':<13} | {'SOURCE FOLDER':<23} | {'GROUND TRUTH VECTOR (L2-Norm)':<30} | {'PREDICTED VECTOR (L2-Norm)':<30} | {'ANGULAR ERROR'}")
    print("="*135)

    with torch.no_grad():
        for idx in range(len(eval_dataset)):
            rgb_tensor, target_tensor, filename, source_folder = eval_dataset[idx]

            # Pack singular array testing dimensions batches
            rgb_in = rgb_tensor.unsqueeze(0).to(device)
            target_in = target_tensor.unsqueeze(0).to(device)

            # Predict using only the single required 'rgb' channel parameter to match your forward implementation
            predicted_tensor = model(rgb_in)

            # Pull L2-normalized components for comparative array analytics tracking
            gt_normalized = F.normalize(target_in, p=2, dim=1).cpu().numpy()[0]
            pred_normalized = F.normalize(predicted_tensor, p=2, dim=1).cpu().numpy()[0]

            # Extract scalar angular divergence metrics performance parameters
            angular_error = calculate_angular_error(predicted_tensor, target_in).item()

            gt_str = f"[{gt_normalized[0]:.4f}, {gt_normalized[1]:.4f}, {gt_normalized[2]:.4f}]"
            pred_str = f"[{pred_normalized[0]:.4f}, {pred_normalized[1]:.4f}, {pred_normalized[2]:.4f}]"

            print(f" {filename:<13} | {source_folder:<23} | {gt_str:<30} | {pred_str:<30} | {angular_error:.2f}°")
    print("="*135 + "\n")

if __name__ == "__main__":
    run_apple_prediction_check()

Target Execution Device: cpu
Loaded trained baseline parameters from: /content/drive/MyDrive/Deep_learning_data/universal_light_direction_model.pth

 IMAGE         | SOURCE FOLDER           | GROUND TRUTH VECTOR (L2-Norm)  | PREDICTED VECTOR (L2-Norm)     | ANGULAR ERROR
 Apple_1.jpg   | Apple/RGB               | [-0.7889, 0.3768, 0.4854]      | [-0.2629, 0.0732, 0.9620]      | 45.41°
 Apple_2.jpg   | Apple/RGB               | [-0.6401, -0.1977, 0.7424]     | [-0.4851, 0.1567, 0.8603]      | 23.33°



In [5]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF
from PIL import Image

# =========================================================
# PATH CONFIGURATION
# =========================================================
DATASET_ROOT = "/content/drive/MyDrive/Deep_learning_data"
DRIVE_SAVE_PATH = os.path.join(DATASET_ROOT, "universal_light_direction_model.pth")

# Switched evaluation domain configurations to look at your primary trained dataset
EVAL_TRAINED_CONFIG = {
    "sample_prediction": {
        "rgb_dir": "sample_prediction/RGB",
        "depth_dir": "sample_prediction/depth",
        "mask_dir": "sample_prediction/test_mask",   # Handled folder name variant 'test_mask' from Drive
        "labels_file": "sample_prediction/light_directions.xlsx"
    }
}

# =========================================================
# METRIC FUNCTION
# =========================================================
def calculate_angular_error(v1, v2):
    v1_norm = F.normalize(v1, p=2, dim=1)
    v2_norm = F.normalize(v2, p=2, dim=1)
    dot_product = torch.sum(v1_norm * v2_norm, dim=1).clamp(-1.0, 1.0)
    return torch.rad2deg(torch.acos(dot_product))

# =========================================================
# UNIVERSAL DATASET PIPELINE (RGB CORE EVALUATION)
# =========================================================
class TrainedEvalDataset(Dataset):
    def __init__(self, config_dict):
        self.all_samples = []

        for domain_name, paths in config_dict.items():
            labels_p = os.path.normpath(os.path.join(DATASET_ROOT, paths["labels_file"]))
            if not os.path.exists(labels_p):
                print(f"Skipping domain '{domain_name}': Reference targets file missing at {labels_p}")
                continue

            if labels_p.endswith('.xlsx') or labels_p.endswith('.xls'):
                df = pd.read_excel(labels_p)
            else:
                df = pd.read_csv(labels_p)

            df.columns = [str(c).strip().lower() for c in df.columns]

            possible_file_cols = ['filename', 'file_name', 'image', 'img', 'name', 'image_name', 'imagename']
            file_col = next((c for c in possible_file_cols if c in df.columns), df.columns[0])

            light_cols = None
            possible_triplets = [
                ['light_x', 'light_y', 'light_z'],
                ['light_dir_x', 'light_dir_y', 'light_dir_z'],
                ['light_direction_x', 'light_direction_y', 'light_direction_z']
            ]
            for triplet in possible_triplets:
                if all(c in df.columns for c in triplet):
                    light_cols = triplet
                    break
            if light_cols is None:
                light_cols = list(df.columns[-3:])

            for idx, row in df.iterrows():
                fname = str(row[file_col]).strip()
                self.all_samples.append({
                    "domain": domain_name,
                    "filename": fname,
                    "target": [float(row[c]) for c in light_cols],
                    "paths": paths
                })

    def find_image_path(self, folder, filename):
        exact = os.path.join(folder, filename)
        if os.path.exists(exact):
            return exact

        base_name = os.path.splitext(filename)[0]
        num_match = re.search(r'\d+', base_name)
        num_str = num_match.group(0) if num_match else None

        candidates = [filename, base_name]
        for ext in ['jpg', 'jpeg', 'png', 'JPG', 'JPEG', 'PNG']:
            candidates.append(f"{base_name}.{ext}")
            if num_str:
                clean_base = base_name.replace(num_str, str(int(num_str)))
                candidates.append(f"{clean_base}.{ext}")

        for cand in set(candidates):
            cand_path = os.path.join(folder, cand)
            if os.path.exists(cand_path):
                return cand_path

        if os.path.exists(folder):
            all_files = os.listdir(folder)
            for f in all_files:
                if base_name.lower() in f.lower():
                    return os.path.join(folder, f)
                if num_str:
                    f_num = re.search(r'\d+', f)
                    if f_num and int(f_num.group(0)) == int(num_str):
                        return os.path.join(folder, f)

        raise FileNotFoundError(f"Missing sample file: {filename} inside folder context: {folder}")

    def __len__(self):
        return len(self.all_samples)

    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        filename = sample["filename"]
        paths = sample["paths"]

        rgb_folder_path = os.path.normpath(os.path.join(DATASET_ROOT, paths["rgb_dir"]))
        rgb_p = self.find_image_path(rgb_folder_path, filename)

        if not os.path.exists(rgb_p):
            raise FileNotFoundError(f"Missing RGB source image file at: {rgb_p}")

        # Process single RGB stream to comply with your trained ResNet18 forward pass signature
        rgb_img = Image.open(rgb_p).convert('RGB').resize((224, 224))
        rgb_tensor = TF.to_tensor(rgb_img)
        rgb_tensor = TF.normalize(rgb_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        target_tensor = torch.tensor(sample["target"], dtype=torch.float32)

        # Extract the true localized filename from Drive disk storage for clarity
        actual_disk_filename = os.path.basename(rgb_p)
        return rgb_tensor, target_tensor, actual_disk_filename, paths["rgb_dir"]

# =========================================================
# RUNNER
# =========================================================
def run_trained_dataset_check():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Execution Device: {device}")

    eval_dataset = TrainedEvalDataset(EVAL_TRAINED_CONFIG)
    if len(eval_dataset) == 0:
        print("Error: Could not locate samples inside sample_prediction/ folder setup.")
        return

    # Call your active workspace class block model definition
    try:
        model = MultiModalRobustModel(output_dim=3).to(device)
    except NameError:
        print("Error: Make sure your 'MultiModalRobustModel' class design block is run first.")
        return

    if os.path.exists(DRIVE_SAVE_PATH):
        model.load_state_dict(torch.load(DRIVE_SAVE_PATH, map_location=device))
        print(f"Loaded trained baseline parameters from: {DRIVE_SAVE_PATH}")
    else:
        print(f"Weights missing at: {DRIVE_SAVE_PATH}")
        return

    model.eval()

    print("\n" + "="*135)
    print(f" {'DISK FILENAME':<22} | {'SOURCE FOLDER':<23} | {'GROUND TRUTH VECTOR (L2-Norm)':<30} | {'PREDICTED VECTOR (L2-Norm)':<30} | {'ANGULAR ERROR'}")
    print("="*135)

    with torch.no_grad():
        # Evaluate up to the first 5 images found within the Excel target list
        for idx in range(min(5, len(eval_dataset))):
            rgb, target, disk_filename, source_folder = eval_dataset[idx]

            rgb_in = rgb.unsqueeze(0).to(device)
            target_in = target.unsqueeze(0).to(device)

            # Executing pure 3-channel forward mapping inference pass
            predicted_tensor = model(rgb_in)

            gt_normalized = F.normalize(target_in, p=2, dim=1).cpu().numpy()[0]
            pred_normalized = F.normalize(predicted_tensor, p=2, dim=1).cpu().numpy()[0]

            angular_error = calculate_angular_error(predicted_tensor, target_in).item()

            gt_str = f"[{gt_normalized[0]:.4f}, {gt_normalized[1]:.4f}, {gt_normalized[2]:.4f}]"
            pred_str = f"[{pred_normalized[0]:.4f}, {pred_normalized[1]:.4f}, {pred_normalized[2]:.4f}]"

            print(f" {disk_filename:<22} | {source_folder:<23} | {gt_str:<30} | {pred_str:<30} | {angular_error:.2f}°")
    print("="*135 + "\n")

if __name__ == "__main__":
    run_trained_dataset_check()

Target Execution Device: cpu
Loaded trained baseline parameters from: /content/drive/MyDrive/Deep_learning_data/universal_light_direction_model.pth

 DISK FILENAME          | SOURCE FOLDER           | GROUND TRUTH VECTOR (L2-Norm)  | PREDICTED VECTOR (L2-Norm)     | ANGULAR ERROR
 cropped_frame_0000.jpg | sample_prediction/RGB   | [-0.8632, 0.0711, 0.4998]      | [-0.8759, 0.0504, 0.4799]      | 1.80°
 cropped_frame_0001.jpg | sample_prediction/RGB   | [-0.3303, -0.0022, 0.9439]     | [-0.3396, -0.0100, 0.9405]     | 0.72°
 cropped_frame_0002.jpg | sample_prediction/RGB   | [0.1956, -0.0756, 0.9778]      | [-0.2961, 0.0939, 0.9505]      | 30.19°
 cropped_frame_0003.jpg | sample_prediction/RGB   | [-0.6074, 0.0178, 0.7942]      | [-0.5470, 0.0003, 0.8371]      | 4.36°
 cropped_frame_0004.jpg | sample_prediction/RGB   | [-0.8314, 0.0400, 0.5543]      | [-0.8677, 0.0654, 0.4928]      | 4.34°

